# NDSI over Pyrenees in Jupytergis with `jupyter-tiler`

To run this notebook, you'll need the dependencies in this repository's `"play"` dependency group.
For most people, installing the right dependencies might look like:

```bash
uv sync --group dev --group test --group play
```

## First, grab some data and do a calculation on it

We want to see that we can display calculated data from an in-memory dataset.

### Get Sentinel 2 data as an Xarray `DataSet`

#### Define a Pyrenees Bounding Box

In [ ]:
from leafmap.plot import bbox_to_gdf

#bbox = [-72.99321, 41.23109, -72.85227, 41.37502]
#bbox = [1.306, 42.527, 1.551, 42.662]
#bbox = [1.0, 42.5, 1.7, 43.1]
bbox = [-1.7, 42.2, 3.0, 43.4]
bbox_to_gdf(bbox).explore()

#### Query the Element84 Earth Search STAC catalog for Sentinel2 data

In [ ]:
from pystac_client import Client

sentinel2_stac_items = (
    Client.open("https://earth-search.aws.element84.com/v1")
    .search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime="2024-01-01/2024-02-01",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    .item_collection()
)
sentinel2_stac_items

#### Load the data into an Xarray `Dataset`

In [ ]:
import odc.stac

sentinel2_dataset = odc.stac.load(
    sentinel2_stac_items,
    bands=["red", "green", "blue", "swir16"],
    chunks={'x': 2048, 'y': 2048},
    bbox=bbox,
    resolution=20,
    groupby="solar_day",
)
sentinel2_dataset

In [ ]:
s2_visible = sentinel2_dataset.max(dim='time')
s2_visible

In [ ]:
config_options = {
                  'AWS_VIRTUAL_HOSTING': 'FALSE',
                  'GDAL_HTTP_UNSAFESSL': 'YES',
                  'AWS_HTTPS': 'YES'}
import rasterio

In [ ]:
worker_env = {
               'AWS_VIRTUAL_HOSTING': 'FALSE',
               'GDAL_HTTP_UNSAFESSL': 'YES',
                  'AWS_HTTPS': 'YES'}

## Dask

In [ ]:
cluster.close()

In [ ]:
from dask_kubernetes.operator import KubeCluster

config = {
   "name": "injhub",
   "namespace": "jhub",
   "image": "guillaumeeb/pangeo-ml-notebook:2026.09.14",
   "n_workers": 2,
   "resources":{"requests": {"memory": "4Gi"}, "limits": {"memory": "4Gi"}}
}

cluster = KubeCluster(**config)

In [ ]:
cluster.scale(20)
cluster

In [ ]:
dask_client = cluster.get_client()
dask_client

In [ ]:
def export_env(env_dict):
    import os
    for key, value in env_dict.items():
        os.environ[key] = value

dask_client.run(export_env, worker_env)

In [ ]:
from distributed.diagnostics.plugin import WorkerPlugin

def configure_workers_environment():
        
    rio_env = rasterio.Env(
        AWS_VIRTUAL_HOSTING='FALSE',
        GDAL_HTTP_UNSAFESSL='YES',
        GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
    )
    rio_env.__enter__()
    return None
    
dask_client.register_worker_callbacks(configure_workers_environment)

### Visualize raw data in RGB

In [ ]:
%%time
with rasterio.Env(**config_options):
    rgb = sentinel2_dataset[["red", "green", "blue"]].to_array(dim="band").max("time")
    rgb_scaled = (rgb / 3000).clip(0, 1)  # Scale and clip for display
    rgb_scaled

In [ ]:
#Visualize subset
with rasterio.Env(**config_options):
    rgb_scaled[:,2000:3000,10000:12000].plot.imshow()

### Calculate NDSI

In [ ]:
%%time
with rasterio.Env(**config_options):
    ndsi = (
        (
            (s2_visible.green - s2_visible.swir16)
            / (s2_visible.green + s2_visible.swir16)
        )
        .where(lambda ndsi: ndsi < 1)
    )

    #remove?
    ndsi = ndsi.persist()
    

In [ ]:
ndsi[2000:3000,10000:12000].plot.imshow()

## Test out `jupyter-tiler`

...with the `ndvi` `DataArray` we just calculated!

In [ ]:
type(ndsi)

In [ ]:
from jupytergis import GISDocument
doc = GISDocument(
    longitude=1.7092461496028497,
    latitude=42.530360648396055,
    zoom=11.38992197518327,
)
await doc.ready()
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png"
)
doc.sidecar(title="Map", anchor="split-right")

In [ ]:
await doc.add_data_array_layer(
    name="NDSI Layer",
    data_array=ndsi,
    colormap_name="viridis",
    colormap_range=(0, 1),
)

In [ ]:
snow = ndsi > 0.4
snow = snow.astype('uint8')
snow

In [ ]:
await doc.add_data_array_layer(
    name="Snow bool",
    data_array=snow,
    colormap_range=(0, 1),
)

## Add another year

In [ ]:
sentinel2_stac_items = (
    Client.open("https://earth-search.aws.element84.com/v1")
    .search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime="2023-01-01/2023-02-01", #Problems with some files in 2022
        query={"eo:cloud_cover": {"lt": 15}},
    )
    .item_collection()
)
sentinel2_stac_items

In [ ]:
sentinel2_dataset_2022 = odc.stac.load(
    sentinel2_stac_items,
    bands=["red", "green", "blue", "swir16"],
    chunks={'x': 2048, 'y': 2048},
    bbox=bbox,
    resolution=20,
    groupby="solar_day",
)
sentinel2_dataset_2022

In [ ]:
s2_visible_2022 = sentinel2_dataset_2022.max(dim='time')
s2_visible_2022

In [ ]:
#del ndsi_2022
with rasterio.Env(**config_options):
    ndsi_2022 = (
        (
            (s2_visible_2022.green - s2_visible_2022.swir16)
            / (s2_visible_2022.green + s2_visible_2022.swir16)
        )
        .where(lambda ndsi_2022: ndsi_2022 < 1)
    )

    #remove?
    ndsi_2022 = ndsi_2022.persist()

In [ ]:
snow_2022 = ndsi_2022 > 0.4
snow_2022 = snow_2022.astype('uint8')
snow_2022

In [ ]:
await doc.add_data_array_layer(
    name="NDSI 2022",
    data_array=ndsi_2022,
    colormap_name="cool",
    colormap_range=(0, 1),
)

In [ ]:
await doc.add_data_array_layer(
    name="Snow 2022",
    data_array=snow_2022,
    colormap_range=(0, 1),
    colormap_name="cool"
)

In [ ]:
cluster.close()